# Dengue bulletin extraction — analysis\n\nLoads `data/extracted/signals.jsonl` into a flat DataFrame. The `signal.*` nested object is flattened into columns so each row is one bulletin.

In [2]:
import json
import pandas as pd
from pathlib import Path

SIGNALS_PATH = Path("data/extracted/signals.jsonl")

records = []
with SIGNALS_PATH.open(encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        row = json.loads(line)
        flat = {k: v for k, v in row.items() if k != "signal"}
        flat.update({f"signal_{k}": v for k, v in row["signal"].items()})
        records.append(flat)

df = pd.DataFrame(records)
df["bulletin_publication_date"] = pd.to_datetime(df["bulletin_publication_date"])
df["extracted_at"] = pd.to_datetime(df["extracted_at"])

print(f"{len(df)} records, {df['signal_is_arbovirus_related'].sum()} arbovirus-related")
df.head()

15 records, 11 arbovirus-related


,source_file,bulletin_publication_date,epi_week_reported,model_id,prompt_version,schema_version,extracted_at,signal_is_arbovirus_related,signal_geographic_scope,signal_uf,signal_reported_trend,signal_alert_level,signal_serotypes_mentioned,signal_emerging_or_new_serotype,signal_forward_looking_warning,signal_regions_of_concern,signal_recommended_action_intensity,signal_evidence_span,signal_requires_human_review
0,2019/be-vol-be-50-no-35-situacao-da-raiva-no-b...,2019-11-19,None,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,2026-05-30 01:16:50.014097+00:00,False,nao_se_aplica,None,nao_informado,nao_informado,[],False,False,[],nao_informado,Sumário: 1 Vigilância Epidemiológica do Saramp...,False
1,2019/boletim-epidemiologico-svs-38-2-interativ...,2019-12-18,None,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,2026-05-30 01:17:10.540328+00:00,True,nacional,None,alta,nao_informado,[],False,False,"[Minas Gerais, Espírito Santo, São Paulo, Acre...",intensificacao,Monitoramento dos casos de arboviroses urbanas...,True
2,2019/boletim-especial-svs-16-anos.pdf,NaT,None,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,2026-05-30 01:17:48.446096+00:00,True,nacional,None,nao_informado,nao_informado,[],False,False,"[Nordeste, Ceará, Rio de Janeiro]",intensificacao,As ações de vigilância e controle de chikungun...,True
3,2020/boletim-epidemiologico-svs-03-v3.pdf,2020-01-22,None,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,2026-05-30 01:17:59.930938+00:00,False,nao_se_aplica,None,nao_informado,nao_informado,[],False,False,[],nao_informado,Identificação de um caso de febre hemorrágica ...,False
4,2020/boletim-epidemiologico-svs-16.pdf,2020-04-16,None,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,2026-05-30 01:18:26.922961+00:00,True,nacional,None,nao_informado,alerta,"[DENV-1, DENV-2, DENV-4]",False,True,"[Acre, São Paulo, Paraná, Mato Grosso do Sul, ...",intensificacao,Monitoramento dos casos de arboviroses urbanas...,True


In [3]:
df.dtypes

source_file                                            str
bulletin_publication_date                   datetime64[us]
epi_week_reported                                   object
model_id                                               str
prompt_version                                         str
schema_version                                         str
extracted_at                           datetime64[us, UTC]
signal_is_arbovirus_related                           bool
signal_geographic_scope                                str
signal_uf                                           object
signal_reported_trend                                  str
signal_alert_level                                     str
signal_serotypes_mentioned                          object
signal_emerging_or_new_serotype                       bool
signal_forward_looking_warning                        bool
signal_regions_of_concern                           object
signal_recommended_action_intensity                    s

## Arbovirus-related bulletins only

In [11]:
df.columns
df[['source_file', 'bulletin_publication_date', 'signal_is_arbovirus_related',
    'signal_uf', 'signal_evidence_span']]

,source_file,bulletin_publication_date,signal_is_arbovirus_related,signal_uf,signal_evidence_span
0,2019/be-vol-be-50-no-35-situacao-da-raiva-no-b...,2019-11-19,False,None,Sumário: 1 Vigilância Epidemiológica do Saramp...
1,2019/boletim-epidemiologico-svs-38-2-interativ...,2019-12-18,True,None,Monitoramento dos casos de arboviroses urbanas...
2,2019/boletim-especial-svs-16-anos.pdf,NaT,True,None,As ações de vigilância e controle de chikungun...
3,2020/boletim-epidemiologico-svs-03-v3.pdf,2020-01-22,False,None,Identificação de um caso de febre hemorrágica ...
4,2020/boletim-epidemiologico-svs-16.pdf,2020-04-16,True,None,Monitoramento dos casos de arboviroses urbanas...
5,2020/boletim-epidemiologico-svs-17.pdf,2020-04-24,True,None,Ativação do COE Arboviroses (Março).
6,2020/boletim-epidemiologico-svs-26.pdf,2020-06-24,False,None,"O boletim trata de Sarampo, Leishmanioses, doe..."
7,2020/boletim-epidemiologico-svs-28-v2.pdf,2020-07-14,True,None,"a partir da SE 12, observa-se uma diminuição d..."
8,2020/boletim-epidemiologico-svs-32.pdf,2020-08-07,True,None,Vacina Febre Amarela. Imunobiológicos com aten...
9,2020/boletim-epidemiologico-vol-51-no-01.pdf,2020-01-16,True,None,O aumento da frequência de epizootias em PNH c...


In [4]:
arbo = df[df["signal_is_arbovirus_related"]].copy()
arbo[["source_file", "bulletin_publication_date", "signal_geographic_scope",
      "signal_reported_trend", "signal_alert_level", "signal_serotypes_mentioned",
      "signal_forward_looking_warning", "signal_recommended_action_intensity",
      "signal_requires_human_review"]]

,source_file,bulletin_publication_date,signal_geographic_scope,signal_reported_trend,signal_alert_level,signal_serotypes_mentioned,signal_forward_looking_warning,signal_recommended_action_intensity,signal_requires_human_review
1,2019/boletim-epidemiologico-svs-38-2-interativ...,2019-12-18,nacional,alta,nao_informado,[],False,intensificacao,True
2,2019/boletim-especial-svs-16-anos.pdf,NaT,nacional,nao_informado,nao_informado,[],False,intensificacao,True
4,2020/boletim-epidemiologico-svs-16.pdf,2020-04-16,nacional,nao_informado,alerta,"[DENV-1, DENV-2, DENV-4]",True,intensificacao,True
5,2020/boletim-epidemiologico-svs-17.pdf,2020-04-24,nacional,nao_informado,nao_informado,"[DENV-1, DENV-2, DENV-4]",False,intensificacao,False
7,2020/boletim-epidemiologico-svs-28-v2.pdf,2020-07-14,nacional,queda,nao_informado,"[DENV-1, DENV-2, DENV-4]",False,intensificacao,True
8,2020/boletim-epidemiologico-svs-32.pdf,2020-08-07,nacional,nao_informado,nao_informado,[],False,rotina,False
9,2020/boletim-epidemiologico-vol-51-no-01.pdf,2020-01-16,nacional,alta,alerta,[],True,intensificacao,False
10,2020/boletim-epidemiologico-vol-51-no-02.pdf,2020-01-16,nacional,alta,nao_informado,[],False,intensificacao,False
12,2020/boletim-epidemiologico-vol-51-no-05.pdf,2020-01-29,nacional,estavel,nao_informado,[],False,intensificacao,True
13,2020/boletim-epidemiologico-vol-51-no-06.pdf,2020-03-06,nacional,nao_informado,alerta,[],True,intensificacao,True


## Categorical distributions

In [5]:
for col in ["signal_reported_trend", "signal_alert_level", "signal_recommended_action_intensity", "signal_geographic_scope"]:
    print(df[col].value_counts().to_string())
    print()

signal_reported_trend
nao_informado    9
alta             3
estavel          2
queda            1

signal_alert_level
nao_informado    12
alerta            3

signal_recommended_action_intensity
intensificacao    9
nao_informado     5
rotina            1

signal_geographic_scope
nacional         11
nao_se_aplica     4



## Provenance check\n\nEvery record must have a consistent model, prompt version, and schema version — otherwise results are not comparable across the corpus.

In [6]:
df[["model_id", "prompt_version", "schema_version"]].value_counts().reset_index()

,model_id,prompt_version,schema_version,count
0,gemini-2.5-flash,2026-05-extraction-v2,1.1.0,15
